In [ ]:
# !pip install torch torchvision scikit-learn matplotlib tqdm -q   # Colab only

import copy, json, os, random, time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import ConcatDataset, DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from tqdm.auto import tqdm

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

## 1. Configuration

In [ ]:
class CFG:
    # --- paths ---
    checkpoint   = "checkpoints/cifar100_resnet18.pt"
    output       = "checkpoints/cifar100_resnet18_w2probe.pt"
    probe_ckpt   = "checkpoints/probe_retain.pt"   # cached across runs
    refit_probe  = False
    data_root    = "~/data"
    plot_dir     = "figures"
    results_json = "results_w2probe.json"

    # --- split ---
    forget_class    = 1            # aquarium fish (fine label)
    forget_fraction = 0.5
    split_seed      = 1234
    seed            = 2025
    cal_fraction    = 0.5
    cal_seed        = 99

    # --- runtime ---
    device      = "cuda:0"
    num_workers = 0 if os.name == "nt" else 8
    use_amp     = True
    epochs      = 1

    # --- optimisation ---
    lr           = 1e-4
    weight_decay = 0.01
    warmup_ratio = 0.10
    clip_grad    = True
    max_norm     = 5.0
    batch_retain = 128
    batch_forget = 64

    # --- objective ---
    lambda_r     = 1.0             # initial lambda (NOT 0 -- retain active at step 0)
    lambda_mode  = "adaptive"      # "fixed" | "adaptive" (dual ascent)
    retain_mode  = "one_sided"     # "one_sided" | "two_sided"
    eps_retain   = -1.0            # <0 -> auto = eps_slack * sampling floor
    eps_slack    = 1.5
    rho0         = 2.0
    beta         = 1.2
    lambda_max   = 100.0
    check_every  = 100
    n_quantiles  = 256             # resolution of frozen reference quantile fns
    cal_pool     = 256             # samples per refresh of sg[P_l(D_cal)]
    cal_refresh  = 10              # steps between refreshes
    cal_ema      = 0.0

    # --- FitCheap probe ---
    probe_size   = 5000
    probe_steps  = 300
    probe_lr     = 1e-3
    probe_wd     = 0.01
    probe_bs     = 128
    probe_seed   = 4242
    probe_reinit = "head_layer4"   # "head" | "head_layer4" | "none"
    probe_bn     = "eval"          # "eval" | "train"

    # --- model selection ---
    select_best = True
    select_n    = 512

    # --- eval / bookkeeping ---
    mia_seed      = 7
    mia_n_members = 2000
    probe_every   = 50
    log_every     = 25
    make_plots    = True

    # --- reproduce the bare equation (disables every refinement) ---
    literal_objective = False


if CFG.literal_objective:
    CFG.lambda_mode = "fixed"
    CFG.retain_mode = "two_sided"
    CFG.cal_refresh = 1
    CFG.cal_pool    = CFG.batch_forget
    CFG.cal_ema     = 0.0
    CFG.select_best = False
    print("literal_objective: all refinements disabled")

DEVICE  = torch.device(CFG.device if torch.cuda.is_available() else "cpu")
USE_AMP = CFG.use_amp and DEVICE.type == "cuda"
os.makedirs(CFG.plot_dir, exist_ok=True)
print(f"device={DEVICE}  amp={USE_AMP}")
print(f"objective: W2^2(Df, sg[Dcal]) + lambda * W2^2(Dr, sg[probe])  "
      f"[lambda={CFG.lambda_r}, mode={CFG.lambda_mode}, retain={CFG.retain_mode}]")

## 2. Constants and the retrained reference

`GOLD` holds the accuracies and MIA efficacies of the retrained model (trained without $D_f$) on this split.
They are used only to report deviations from the counterfactual; the method never reads them.

In [ ]:
SUPERCLASS_MAPPING = [
    4, 1, 14, 8, 0, 6, 7, 7, 18, 3,
    3, 14, 9, 18, 7, 11, 3, 9, 7, 11,
    6, 11, 5, 10, 7, 6, 13, 15, 3, 15,
    0, 11, 1, 10, 12, 14, 16, 9, 11, 5,
    5, 19, 8, 8, 15, 13, 14, 17, 18, 10,
    16, 4, 17, 4, 2, 0, 17, 4, 18, 17,
    10, 3, 2, 12, 12, 16, 12, 1, 9, 19,
    2, 10, 0, 1, 16, 12, 9, 13, 15, 13,
    16, 19, 2, 4, 6, 19, 5, 5, 8, 19,
    18, 1, 2, 15, 6, 0, 17, 8, 14, 13,
]

_CIFAR_MEAN = (0.5070751592371323, 0.48654887331495095, 0.4409178433670343)
_CIFAR_STD  = (0.2673342858792401, 0.2564384629170883, 0.27615047132568404)

DEFAULT_TRANSFORM = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(_CIFAR_MEAN, _CIFAR_STD),
])

GOLD = {"Df": 81.20, "Dadj": 99.60, "Drem": 98.94, "fish_te": 78.00,
        "Drem_te": 84.48, "mia_Df": 0.376, "mia_Dadj": 0.072}

PRIOR_RUNS = {}   # optionally add rows (name -> 7 numbers) to compare against


## 3. Data

In [ ]:
class CIFAR100SuperClassDataset(Dataset):
    def __init__(self, base):
        self.base = base
    def __len__(self):
        return len(self.base)
    def __getitem__(self, i):
        img, fine_label = self.base[i]
        return img, SUPERCLASS_MAPPING[fine_label]


def _base(train, data_root, download=True):
    return datasets.CIFAR100(root=os.path.expanduser(data_root), train=train,
                             download=download, transform=DEFAULT_TRANSFORM)


def instance_split_indices(base, forget_class_index, forget_fraction, split_seed):
    targets = np.asarray(base.targets)
    cls_idx = np.where(targets == forget_class_index)[0]
    rng = np.random.default_rng(split_seed)
    perm = rng.permutation(cls_idx)
    n_forget = int(round(forget_fraction * len(cls_idx)))
    return (np.sort(perm[:n_forget]), np.sort(perm[n_forget:]),
            np.setdiff1d(np.arange(len(targets)), cls_idx))


def make_loader(base, indices, bs, shuffle, num_workers=None, drop_last=False):
    nw = CFG.num_workers if num_workers is None else num_workers
    ds = CIFAR100SuperClassDataset(Subset(base, np.asarray(indices).tolist()))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=nw,
                      pin_memory=True, drop_last=drop_last)


def fish_calibration_loaders(forget_class_index, cal_fraction, cal_seed, bs, data_root):
    '''D_cal: held-out images of the *forget class*, split off from the CIFAR test
    set so they were never in S. The complementary half stays untouched for
    evaluation (fish(te)).'''
    base = _base(False, data_root)
    targets = np.asarray(base.targets)
    cls_idx = np.where(targets == forget_class_index)[0]
    rng = np.random.default_rng(cal_seed)
    perm = rng.permutation(cls_idx)
    n_cal = round(cal_fraction * len(cls_idx))
    cal_idx, eval_idx = np.sort(perm[:n_cal]), np.sort(perm[n_cal:])
    rest_idx = np.where(targets != forget_class_index)[0]
    return (make_loader(base, cal_idx, bs, True),
            make_loader(base, eval_idx, 256, False),
            make_loader(base, rest_idx, 256, False))


def stratified_subsample(base, indices, size, seed):
    '''Class-balanced subsample of D_r, balanced over the 20 superclasses.

    A proportional draw would put only ~size * 250/49750 adjacent (fish) samples
    in the probe, leaving the fish region of the
    reference unanchored, which is exactly where entanglement lives. Balancing
    over superclasses guarantees a full per-class quota instead.'''
    indices = np.asarray(indices)
    fine = np.asarray(base.targets)[indices]
    sup = np.asarray(SUPERCLASS_MAPPING)[fine]
    classes = np.unique(sup)
    per_class = max(1, int(round(size / len(classes))))
    rng = np.random.default_rng(seed)
    picked = []
    for cl in classes:
        pool = indices[sup == cl]
        picked.append(rng.choice(pool, min(per_class, len(pool)), replace=False))
    return np.sort(np.concatenate(picked))


def cycle(loader):
    while True:
        for batch in loader:
            yield batch


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CFG.seed)

## 4. Evaluation, MIA and diagnostic helpers

In [ ]:
def _with_progress(loader, desc):
    return (tqdm(loader, desc=desc, unit="batch", leave=False)
            if desc and len(loader) > 1 else loader)


@torch.no_grad()
def acc_and_loss(model, loader, device, desc=None):
    was_training = model.training
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    for images, labels in _with_progress(loader, desc):
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss_sum += nn.CrossEntropyLoss(reduction="sum")(logits, labels).item()
        correct  += logits.argmax(1).eq(labels).sum().item()
        total    += labels.size(0)
    if was_training:
        model.train()
    if total == 0:                      # guards the empty-loader edge case
        return {"acc": float("nan"), "loss": float("nan")}
    return {"acc": 100.0 * correct / total, "loss": loss_sum / total}


def evaluate_all(model, loaders, device, tag=""):
    out = {k: acc_and_loss(model, v, device, desc=f"{tag}: {k}")
           for k, v in loaders.items()}
    if tag:
        print(f"[{tag}] " + "  ".join(f"{k}={v['acc']:.2f}" for k, v in out.items()))
    return out


@torch.no_grad()
def per_sample_losses(model, loader, device, as_tensor=False, desc=None):
    was_training = model.training
    model.eval()
    vals = []
    for images, labels in _with_progress(loader, desc):
        images, labels = images.to(device), labels.to(device)
        vals.append(F.cross_entropy(model(images), labels, reduction="none").float().cpu())
    if was_training:
        model.train()
    out = torch.cat(vals)
    return out if as_tensor else out.numpy()


def loss_auc(model, loader_a, loader_b, device):
    la = per_sample_losses(model, loader_a, device)
    lb = per_sample_losses(model, loader_b, device)
    y = np.concatenate([np.ones_like(la), np.zeros_like(lb)])
    return float(roc_auc_score(y, np.concatenate([la, lb])))


def fit_mia_attacker(model, member_loader, nonmember_loader, device,
                     n_members=2000, seed=7):
    mem = per_sample_losses(model, member_loader, device)
    non = per_sample_losses(model, nonmember_loader, device)
    rng = np.random.default_rng(seed)
    n = min(n_members, len(mem), len(non))
    mem, non = rng.choice(mem, n, replace=False), rng.choice(non, n, replace=False)
    X = np.concatenate([mem, non]).reshape(-1, 1)
    y = np.concatenate([np.ones(n), np.zeros(n)])
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                              random_state=seed, stratify=y)
    clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    return clf, clf.score(X_te, y_te)


def mia_efficacy(clf, model, loader, device):
    losses = per_sample_losses(model, loader, device).reshape(-1, 1)
    return float((clf.predict(losses) == 0).mean())


def w2_numpy(a, b, n_quantiles=256):
    qs = (np.arange(n_quantiles) + 0.5) / n_quantiles
    return float(np.mean((np.quantile(a, qs) - np.quantile(b, qs)) ** 2))


def ks_numpy(a, b):
    grid = np.concatenate([a, b])
    fa = np.searchsorted(np.sort(a), grid, side="right") / len(a)
    fb = np.searchsorted(np.sort(b), grid, side="right") / len(b)
    return float(np.max(np.abs(fa - fb)))

## 5. 1-D Wasserstein machinery

$$\widehat{W_2^2}(x,\mathrm{ref}) = \frac{1}{n}\sum_i \Big(x_{(i)} - Q_{\mathrm{ref}}\big(\tfrac{i+0.5}{n}\big)\Big)^2$$

Sorting $x$ is a permutation, so all $n$ batch samples receive gradient. References are frozen into a
quantile vector once (`reference_quantiles`) rather than recomputed from raw samples every step.

**One-sided retain term.** With `one_sided=True` only losses *above* the reference are penalised. The probe is a
deliberately cheap model, so its losses sit above $\theta_0$'s; a two-sided term would drag retain accuracy
*down* to meet it. One-sided leaves the model free to be better than the probe.

**Sampling floor (`estimate_w2_floor`).** A finite mini-batch never matches its parent distribution, so the term
has a strictly positive floor. A tolerance $\epsilon_r$ below the floor is violated by sampling noise alone and
$\lambda$ diverges, so $\epsilon_r$ is set relative to the bootstrap floor.

In [ ]:
def midpoint_grid(n, device, dtype=torch.float32):
    '''(i + 0.5)/n quantile levels: avoids the noisy 0/1 order statistics.'''
    return (torch.arange(n, device=device, dtype=dtype) + 0.5) / n


def reference_quantiles(samples, n_quantiles, device):
    '''Freeze a sample set into a fixed quantile function (the sg[.] side).'''
    x = samples.detach().float().to(device).flatten()
    qs = midpoint_grid(n_quantiles, device, x.dtype)
    return torch.quantile(x, qs).contiguous()


def w2_sq_to_reference(losses, ref_q, one_sided=False):
    '''W2^2( empirical(losses), reference ). `losses` keeps its graph;
    `ref_q` is a detached quantile vector.'''
    x = losses.float().flatten()
    n = x.numel()
    xs, _ = torch.sort(x)
    qs = midpoint_grid(n, x.device, x.dtype)
    ref = torch.quantile(ref_q, qs) if ref_q.numel() != n else ref_q
    diff = xs - ref
    if one_sided:
        diff = torch.relu(diff)
    return torch.mean(diff ** 2)


def w2_sq_between(losses_a, losses_b, one_sided=False):
    '''W2^2 where side B is detached in place (stop-gradient on D_cal).'''
    b = losses_b.detach().float().flatten()
    a = losses_a.float().flatten()
    n = a.numel()
    xs, _ = torch.sort(a)
    ref = (torch.sort(b)[0] if b.numel() == n
           else torch.quantile(b, midpoint_grid(n, a.device, a.dtype)))
    diff = xs - ref
    if one_sided:
        diff = torch.relu(diff)
    return torch.mean(diff ** 2)


@torch.no_grad()
def estimate_w2_floor(ref_samples, batch_size, n_quantiles, one_sided=False,
                      n_boot=200, seed=0):
    '''Irreducible W2^2 between a size-`batch_size` draw and its own reference.'''
    x = ref_samples.detach().float().flatten()
    ref_q = reference_quantiles(x, n_quantiles, x.device)
    g = torch.Generator(device="cpu").manual_seed(seed)
    vals = []
    for _ in tqdm(range(n_boot), desc="estimate W2 floor", unit="sample", leave=False):
        idx = torch.randint(0, x.numel(), (batch_size,), generator=g).to(x.device)
        vals.append(w2_sq_to_reference(x[idx], ref_q, one_sided).item())
    return float(np.median(vals))

**Sanity check** (runs without the checkpoint or dataset) — one-sided vs two-sided, and the sampling floor.

In [ ]:
_ref = torch.distributions.Gamma(2.0, 2.0).sample((5000,))
_rq  = reference_quantiles(_ref, 256, _ref.device)
_bat = torch.distributions.Gamma(2.0, 2.0).sample((128,))

for name, x in (("matched", _bat), ("better than probe (x0.3)", _bat * 0.3),
                ("worse than probe (x3.0)", _bat * 3.0)):
    two = w2_sq_to_reference(x, _rq, one_sided=False).item()
    one = w2_sq_to_reference(x, _rq, one_sided=True).item()
    print(f"{name:26s} two_sided={two:8.4f}   one_sided={one:8.4f}")

print(f"\nfloor two_sided (batch=128): {estimate_w2_floor(_ref, 128, 256, False):.5f}")
print(f"floor one_sided (batch=128): {estimate_w2_floor(_ref, 128, 256, True):.5f}")
print("-> 'better than probe' costs 0 one-sided: the model may outperform the probe.")

## 6. FitCheap — the retain probe $\theta_r^{probe}$

The classifier head is re-initialised before fitting, so the probe does not inherit $\theta_0$'s direct
$D_f\to$ logit pathway; the head (and optionally the last stage, `layer4`) is then trained for a few hundred
steps on $D_r^{probe}$ only.

The rest of the backbone stays frozen, so this is a cheap shortcut toward $\theta^{retrain}$, not a claim of
full independence from $\theta_0$: the frozen features were learned with $D_f$ present.

In [ ]:
def _last_linear(model):
    name, module = None, None
    for n_, m in model.named_modules():
        if isinstance(m, nn.Linear):
            name, module = n_, m
    if module is None:
        raise RuntimeError("no nn.Linear head found in the checkpoint")
    return name, module


def fit_cheap_probe(base_model, probe_loader, device, steps=300, lr=1e-3,
                    weight_decay=0.01, reinit="head_layer4", bn_mode="eval",
                    use_amp=True, log_every=50, max_norm=5.0):
    '''A few hundred cheap steps on D_r^probe only. Never sees D_f.

    reinit:  head        - reset the classifier, train the classifier only
             head_layer4 - reset the classifier, train classifier + layer4
             none        - keep theta_0's head, fine-tune it on D_r^probe
    '''
    probe = copy.deepcopy(base_model).to(device)
    head_name, head = _last_linear(probe)
    if reinit in ("head", "head_layer4"):
        head.reset_parameters()

    for p in probe.parameters():
        p.requires_grad_(False)
    trainable = list(head.parameters())
    for p in trainable:
        p.requires_grad_(True)
    if reinit == "head_layer4" and hasattr(probe, "layer4"):
        for p in probe.layer4.parameters():
            p.requires_grad_(True)
            trainable.append(p)

    probe.train() if bn_mode == "train" else probe.eval()
    opt = optim.AdamW(trainable, lr=lr, weight_decay=weight_decay)
    sched = CosineAnnealingLR(opt, T_max=max(steps, 1), eta_min=lr * 1e-2)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    stream = cycle(probe_loader)

    print(f"[probe] head='{head_name}'  trainable="
          f"{sum(p.numel() for p in trainable)/1e6:.2f}M  steps={steps}  "
          f"reinit={reinit}  bn={bn_mode}")

    for s in tqdm(range(steps), desc="FitCheap(D_r^probe)"):
        x, y = next(stream)
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            loss = F.cross_entropy(probe(x), y.long())
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(trainable, max_norm)
        scaler.step(opt); scaler.update(); sched.step()
        if s % log_every == 0:
            print(f"  [probe {s:4d}] ce={loss.item():.4f}")

    probe.eval()
    for p in probe.parameters():
        p.requires_grad_(False)
    return probe

## 7. The unlearning loop

**Pooled calibration reference.** $\mathrm{sg}[P_\ell(D_{cal};\theta)]$ is stop-gradient by construction, so it is
computed under `no_grad` over a pool larger than one mini-batch at no graph cost. Pooling 256 samples and
refreshing every 10 steps stabilises the forget term and damps the feedback loop in which a model degrading on
all fish raises its own target.

**Dual ascent on $\lambda$.** $\lambda$ starts at 1.0 so the retain term is active from step 0, and is updated as
$\lambda \leftarrow \min(\lambda + \rho\,[W_{2,r}^2-\epsilon_r]_+,\ \lambda_{max})$, with $\rho\!\leftarrow\!\beta\rho$
on persistent violation.

**Checkpoint selection** on $W_{2,f}^2 + \lambda_r W_{2,r}^2$ over held-out slices of data the objective already
sees ($D_f$, $D_{cal}$, $D_r^{probe}$). It never consults the retrained model. Set `CFG.select_best = False`
to return the last step instead.

In [ ]:
def run_w2_unlearn(model, device, forget_loader, retain_loader, cal_loader,
                   probe_ref_q, eps_retain, eval_loaders=None, select_loaders=None):
    model.eval()
    model_unlearn = copy.deepcopy(model)
    model_unlearn.eval()          # BN stats frozen: unlearning must not rewrite them

    cycled_forget, cycled_cal = cycle(forget_loader), cycle(cal_loader)
    optimizer = optim.AdamW(model_unlearn.parameters(), lr=CFG.lr,
                            weight_decay=CFG.weight_decay)

    total_steps  = CFG.epochs * len(retain_loader)
    warmup_steps = max(int(total_steps * CFG.warmup_ratio), 1)
    scheduler = SequentialLR(
        optimizer,
        schedulers=[LinearLR(optimizer, start_factor=0.01, end_factor=1.0,
                             total_iters=warmup_steps),
                    CosineAnnealingLR(optimizer,
                                      T_max=max(total_steps - warmup_steps, 1),
                                      eta_min=1e-7)],
        milestones=[warmup_steps])

    lambda_ = torch.tensor(float(CFG.lambda_r), device=device)
    lam_cap = torch.tensor(float(CFG.lambda_max), device=device)
    rho = CFG.rho0
    one_sided = (CFG.retain_mode == "one_sided")
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    diag = {k: [] for k in ("step", "loss", "W2_f", "W2_r", "mean_l_f",
                            "mean_l_cal", "mean_l_r", "lambda", "rho", "lr")}
    plog = {"step": [], "Df": [], "Dadj": [], "fish": [], "score": []}
    cal_ref = None
    best = {"score": float("inf"), "step": -1, "state": None}
    step = 0

    @torch.no_grad()
    def refresh_cal_reference(prev):
        pool, got = [], 0
        while got < CFG.cal_pool:
            xb, yb = next(cycled_cal)
            xb, yb = xb.to(device), yb.to(device)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                lb = F.cross_entropy(model_unlearn(xb), yb.long(), reduction="none")
            pool.append(lb.float()); got += lb.numel()
        q = reference_quantiles(torch.cat(pool), CFG.n_quantiles, device)
        if prev is not None and CFG.cal_ema > 0.0:
            q = CFG.cal_ema * prev + (1.0 - CFG.cal_ema) * q
        return q

    @torch.no_grad()
    def selection_score():
        l_f = per_sample_losses(model_unlearn, select_loaders["Df"], device, as_tensor=True).to(device)
        l_c = per_sample_losses(model_unlearn, select_loaders["cal"], device, as_tensor=True).to(device)
        l_r = per_sample_losses(model_unlearn, select_loaders["Rprobe"], device, as_tensor=True).to(device)
        w_f = w2_sq_between(l_f, l_c).item()
        w_r = w2_sq_to_reference(l_r, probe_ref_q, one_sided=one_sided).item()
        return w_f + float(CFG.lambda_r) * w_r, w_f, w_r

    progress = tqdm(total=total_steps, desc="w2-unlearn", unit="step")
    for epoch in range(CFG.epochs):
        for batch_id, (x_r, y_r) in enumerate(
                tqdm(retain_loader, desc=f"epoch {epoch+1}/{CFG.epochs}",
                     unit="batch", leave=False)):
            x_r, y_r = x_r.to(device, non_blocking=True), y_r.to(device, non_blocking=True)
            x_f, y_f = next(cycled_forget)
            x_f, y_f = x_f.to(device, non_blocking=True), y_f.to(device, non_blocking=True)

            if cal_ref is None or step % CFG.cal_refresh == 0:
                cal_ref = refresh_cal_reference(cal_ref)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                l_r = F.cross_entropy(model_unlearn(x_r), y_r.long(), reduction="none")
                l_f = F.cross_entropy(model_unlearn(x_f), y_f.long(), reduction="none")

            # W2 terms in fp32: sorting/quantiles in fp16 are badly conditioned
            W2_f = w2_sq_to_reference(l_f, cal_ref)
            W2_r = w2_sq_to_reference(l_r, probe_ref_q, one_sided=one_sided)
            loss = W2_f + lambda_ * W2_r

            scaler.scale(loss).backward()
            if CFG.clip_grad:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model_unlearn.parameters(), CFG.max_norm)
            scale_before = scaler.get_scale()
            scaler.step(optimizer); scaler.update()
            if not (scale_before > scaler.get_scale()):
                scheduler.step()

            if CFG.lambda_mode == "adaptive":
                with torch.no_grad():
                    violation = torch.relu(W2_r.detach() - eps_retain)
                    lambda_ = torch.minimum(lambda_ + rho * violation, lam_cap)
                if batch_id > 0 and batch_id % CFG.check_every == 0 and W2_r.item() > eps_retain:
                    rho *= CFG.beta
                    print(f"  retain constraint violated (W2_r={W2_r.item():.4f} "
                          f"> {eps_retain:.4f}); rho -> {rho:.2f}")

            for k, v in (("step", step), ("loss", loss.item()),
                         ("W2_f", W2_f.item()), ("W2_r", W2_r.item()),
                         ("mean_l_f", l_f.mean().item()),
                         ("mean_l_cal", cal_ref.mean().item()),
                         ("mean_l_r", l_r.mean().item()),
                         ("lambda", float(lambda_.item())), ("rho", rho),
                         ("lr", optimizer.param_groups[0]["lr"])):
                diag[k].append(v)

            if eval_loaders is not None and step % CFG.probe_every == 0:
                plog["step"].append(step)
                for key, name in (("Df", "Df(tr)"), ("Dadj", "Dadj(tr)"), ("fish", "fish(te)")):
                    plog[key].append(acc_and_loss(model_unlearn, eval_loaders[name], device)["acc"])
                if CFG.select_best and select_loaders is not None:
                    sc, w_f, w_r = selection_score()
                    plog["score"].append(sc)
                    if sc < best["score"]:
                        best = {"score": sc, "step": step,
                                "state": copy.deepcopy(model_unlearn.state_dict())}
                        print(f"  [select] new best @step {step}: score={sc:.4f} "
                              f"(W2_f={w_f:.4f}, W2_r={w_r:.4f})")
                else:
                    plog["score"].append(float("nan"))
                model_unlearn.eval()

            if batch_id % CFG.log_every == 0:
                print(f"Batch [{batch_id}] loss={loss.item():.4f} "
                      f"W2_f={W2_f.item():.4f} W2_r={W2_r.item():.4f} "
                      f"l_f={l_f.mean().item():.3f} l_cal={cal_ref.mean().item():.3f} "
                      f"l_r={l_r.mean().item():.3f} lam={lambda_.item():.2f}")
            progress.update(1)
            progress.set_postfix(loss=f"{loss.item():.3f}", W2_f=f"{W2_f.item():.3f}",
                                 W2_r=f"{W2_r.item():.3f}")
            step += 1

    progress.close()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if CFG.select_best and best["state"] is not None:
        print(f"restoring best checkpoint from step {best['step']} (score={best['score']:.4f})")
        model_unlearn.load_state_dict(best["state"])
    return model_unlearn, diag, plog, best

## 8. Plots and reporting

In [ ]:
def save_diagnostics_plot(diag, plog, eps_retain, probe_ref_mean, path):
    s = diag["step"]
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))

    ax = axes[0, 0]
    ax.plot(s, diag["mean_l_f"], color="tab:red", lw=1, label="mean loss $D_f$")
    ax.plot(s, diag["mean_l_cal"], color="tab:green", lw=1,
            label="mean loss $D_{cal}$ (sg, never seen)")
    ax.plot(s, diag["mean_l_r"], color="tab:blue", lw=1, label="mean loss $D_r$")
    ax.axhline(probe_ref_mean, color="tab:purple", ls="--", lw=1, label="probe ref mean")
    ax.set_title("Counterfactual matching: do $D_f$ and $D_{cal}$ converge?")
    ax.set_xlabel("step"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = axes[0, 1]
    ax.plot(s, diag["W2_f"], color="tab:purple", lw=1)
    ax.set_yscale("log")
    ax.set_title("Forget term $W_2^2(P_\\ell(D_f), sg[P_\\ell(D_{cal})])$")
    ax.set_xlabel("step"); ax.grid(alpha=0.3)

    ax = axes[1, 0]
    ax.plot(s, diag["W2_r"], color="tab:blue", lw=1, label="$W_2^2$ retain vs probe")
    ax.axhline(eps_retain, color="gray", ls="--", label=f"$\\epsilon_r$={eps_retain:.4f}")
    ax.set_yscale("log")
    ax2 = ax.twinx()
    ax2.plot(s, diag["lambda"], color="tab:orange", lw=1, alpha=.7)
    ax2.set_ylabel("$\\lambda$", color="tab:orange")
    ax.set_title("Retain anchor (probe) and dual variable")
    ax.set_xlabel("step"); ax.legend(loc="upper left", fontsize=8); ax.grid(alpha=0.3)

    ax = axes[1, 1]
    ax.plot(plog["step"], plog["Df"], marker="^", label="$D_f$ (train)")
    ax.plot(plog["step"], plog["Dadj"], marker="s", label="$D_r^{adj}$ (train)")
    ax.plot(plog["step"], plog["fish"], marker="o", label="fish test (population)")
    ax.axhline(GOLD["Df"], color="tab:red", ls=":", label=f"gold $D_f$={GOLD['Df']}")
    ax.axhline(GOLD["fish_te"], color="tab:green", ls=":",
               label=f"gold fish(te)={GOLD['fish_te']}")
    ax.set_title("Accuracy vs the retrain reference")
    ax.set_xlabel("step"); ax.set_ylabel("accuracy (%)")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plt.tight_layout(); plt.savefig(path, dpi=120); plt.show()
    print(f"saved {path}")


def save_loss_hist_plot(lf1, la1, lu1, lr_probe, lr_ref, path):
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    axes[0].hist(lf1, bins=30, alpha=.6, density=True, color="tab:red", label="$D_f$")
    axes[0].hist(la1, bins=30, alpha=.6, density=True, color="tab:blue",
                 label="$D_r^{adj}$ (member)")
    axes[0].set_title("Memorization axis")
    axes[1].hist(lf1, bins=30, alpha=.6, density=True, color="tab:red", label="$D_f$")
    axes[1].hist(lu1, bins=30, alpha=.6, density=True, color="tab:green",
                 label="unseen fish (counterfactual)")
    axes[1].set_title("Counterfactual axis, overlap = success")
    axes[2].hist(lr_probe, bins=30, alpha=.6, density=True, color="tab:blue",
                 label="$D_r^{probe}$ under $\\theta$")
    axes[2].hist(lr_ref, bins=30, alpha=.6, density=True, color="tab:purple",
                 label="$D_r^{probe}$ under $\\theta_r^{probe}$")
    axes[2].set_title("Retain anchor, overlap = preserved")
    for ax in axes:
        ax.set_xlabel("Per-sample CE loss"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    axes[0].set_ylabel("Density")
    plt.tight_layout(); plt.savefig(path, dpi=120); plt.show()
    print(f"saved {path}")


def print_comparison(results, mia, timings, key="w2probe", label="Swift (this)"):
    table = dict(PRIOR_RUNS)
    table["retrain(gold)"] = (GOLD["Df"], GOLD["Dadj"], GOLD["Drem"], GOLD["fish_te"],
                              GOLD["Drem_te"], GOLD["mia_Df"], GOLD["mia_Dadj"])
    r, m = results[key], mia[key]
    table[label] = (r["Df(tr)"]["acc"], r["Dadj(tr)"]["acc"], r["Drem(tr)"]["acc"],
                    r["fish(te)"]["acc"], r["Drem(te)"]["acc"], m["Df"], m["Dadj"])

    hdr = ("Df(tr)", "Dadj(tr)", "Drem(tr)", "fish(te)", "Drem(te)")
    print("\nAccuracy (%)")
    print(f"{'Model':16s}" + "".join(f"{h:>10s}" for h in hdr))
    print("-" * (16 + 10 * len(hdr)))
    for k, v in table.items():
        print(f"{k:16s}" + "".join(f"{x:10.2f}" for x in v[:5]))

    print("\nMIA efficacy")
    print(f"{'Model':16s}{'Df':>9s}{'Dadj':>9s}{'gap':>9s}")
    print("-" * 43)
    for k, v in table.items():
        print(f"{k:16s}{v[5]:9.3f}{v[6]:9.3f}{v[5]-v[6]:9.3f}")

    print("\nDeviation from the counterfactual (retrain gold), lower is better")
    g = table["retrain(gold)"]
    print(f"{'Model':16s}{'|dDf|':>8s}{'|dAdj|':>8s}{'|dFish|':>9s}"
          f"{'|dMIA_Df|':>11s}{'L1(acc)':>9s}")
    print("-" * 62)
    for k, v in table.items():
        if k == "retrain(gold)":
            continue
        d = [abs(v[i] - g[i]) for i in range(5)]
        print(f"{k:16s}{d[0]:8.2f}{d[1]:8.2f}{d[3]:9.2f}"
              f"{abs(v[5]-g[5]):11.3f}{sum(d):9.2f}")

    print("\nRuntime (s)")
    print(f"{'Stage':28s}{'seconds':>10s}")
    print("-" * 38)
    for k, v in timings.items():
        print(f"{k:28s}{v:10.1f}")
    print("-" * 38)
    print(f"{'Swift total (probe+opt)':28s}"
          f"{timings.get('probe_fit', 0) + timings.get('optimisation', 0):10.1f}")
    print(f"{'retrain reference (paper)':28s}{926.0:10.1f}")

## 9. Load the model and build the splits

In [ ]:
try:
    model = torch.load(CFG.checkpoint, map_location=DEVICE, weights_only=False)
except TypeError:
    model = torch.load(CFG.checkpoint, map_location=DEVICE)
model.to(DEVICE).eval()

train_base = _base(True, CFG.data_root)
f_idx, a_idx, r_idx = instance_split_indices(
    train_base, CFG.forget_class, CFG.forget_fraction, CFG.split_seed)

ev_forget   = make_loader(train_base, f_idx, 256, False)
ev_adjacent = make_loader(train_base, a_idx, 256, False)
ev_remote   = make_loader(train_base, r_idx, 256, False)

cal_loader, ev_fish_holdout, ev_test_remote = fish_calibration_loaders(
    CFG.forget_class, CFG.cal_fraction, CFG.cal_seed, CFG.batch_forget, CFG.data_root)

retain_loader = DataLoader(ConcatDataset([ev_remote.dataset, ev_adjacent.dataset]),
                           batch_size=CFG.batch_retain, shuffle=True,
                           num_workers=CFG.num_workers, pin_memory=True, drop_last=True)
forget_loader = DataLoader(ev_forget.dataset, batch_size=CFG.batch_forget, shuffle=True,
                           num_workers=CFG.num_workers, pin_memory=True, drop_last=True)

eval_loaders = {"Df(tr)": ev_forget, "Dadj(tr)": ev_adjacent, "Drem(tr)": ev_remote,
                "fish(te)": ev_fish_holdout, "Drem(te)": ev_test_remote}

# D_r^probe
retain_idx = np.concatenate([r_idx, a_idx])
probe_idx  = stratified_subsample(train_base, retain_idx, CFG.probe_size, CFG.probe_seed)
probe_train_loader   = make_loader(train_base, probe_idx, CFG.probe_bs, True)
probe_eval_loader    = make_loader(train_base, probe_idx, 256, False)
probe_holdout_idx    = np.setdiff1d(retain_idx, probe_idx)[:2000]
probe_holdout_loader = make_loader(train_base, probe_holdout_idx, 256, False)

n_adj_in_probe = int(np.isin(probe_idx, a_idx).sum())
print(f"Df={len(ev_forget.dataset)}  Dadj={len(ev_adjacent.dataset)}  "
      f"Drem={len(ev_remote.dataset)}  Dr^probe={len(probe_idx)}")
print(f"adjacent (fish) samples inside D_r^probe = {n_adj_in_probe}")
print(f"calibration (in objective) = {len(cal_loader.dataset)} fish test images")
print(f"fish(te) evaluation (untouched) = {len(ev_fish_holdout.dataset)}")

timings, results, mia = {}, {}, {}

## 10. Reference: the original model

In [ ]:
def run_mia(m, tag):
    clf, holdout = fit_mia_attacker(m, ev_remote, ev_test_remote, DEVICE,
                                    n_members=CFG.mia_n_members, seed=CFG.mia_seed)
    eff_f  = mia_efficacy(clf, m, ev_forget, DEVICE)
    eff_a  = mia_efficacy(clf, m, ev_adjacent, DEVICE)
    auc    = loss_auc(m, ev_forget, ev_adjacent, DEVICE)
    auc_cf = loss_auc(m, ev_forget, ev_fish_holdout, DEVICE)
    print(f"[{tag}] atk={holdout:.3f} | efficacy Df={eff_f:.3f} (gold {GOLD['mia_Df']:.3f}) "
          f"Dadj={eff_a:.3f} (gold {GOLD['mia_Dadj']:.3f}) | AUC(Df,Dadj)={auc:.3f} "
          f"| AUC(Df,unseen)={auc_cf:.3f}")
    return {"attacker_acc": holdout, "Df": eff_f, "Dadj": eff_a,
            "auc_adj": auc, "auc_unseen": auc_cf}

results["original"] = evaluate_all(model, eval_loaders, DEVICE, "original")
mia["original"]     = run_mia(model, "original")

## 11. FitCheap

Three diagnostics are printed:

- **probe CE trajectory** — should fall sharply from the reset head, not sit flat.
- **probe vs $\theta_0$ on the same subset** — if the probe's loss is *much* higher, a two-sided retain term
  would pull retain accuracy down; the warning fires and `one_sided` is the guard.
- **probe on unseen retain** — how well the probe generalises, i.e. whether it is a usable reference at all.

In [ ]:
t0 = time.perf_counter()
probe_model = None
if (not CFG.refit_probe) and CFG.probe_ckpt and os.path.exists(CFG.probe_ckpt):
    try:
        blob = torch.load(CFG.probe_ckpt, map_location=DEVICE, weights_only=False)
        if blob.get("probe_idx_hash") == int(probe_idx.sum()):
            probe_model = copy.deepcopy(model)
            probe_model.load_state_dict(blob["state_dict"])
            probe_model.to(DEVICE).eval()
            print(f"loaded cached probe <- {CFG.probe_ckpt}")
        else:
            print("cached probe was fitted on a different subset; refitting")
    except Exception as exc:
        print(f"could not reuse probe cache ({exc}); refitting")

if probe_model is None:
    probe_model = fit_cheap_probe(model, probe_train_loader, DEVICE,
                                  steps=CFG.probe_steps, lr=CFG.probe_lr,
                                  weight_decay=CFG.probe_wd, reinit=CFG.probe_reinit,
                                  bn_mode=CFG.probe_bn, use_amp=USE_AMP,
                                  max_norm=CFG.max_norm)
    if CFG.probe_ckpt:
        os.makedirs(os.path.dirname(CFG.probe_ckpt) or ".", exist_ok=True)
        torch.save({"state_dict": probe_model.state_dict(),
                    "probe_idx_hash": int(probe_idx.sum())}, CFG.probe_ckpt)
        print(f"saved probe -> {CFG.probe_ckpt}")
timings["probe_fit"] = time.perf_counter() - t0

probe_ref_losses = per_sample_losses(probe_model, probe_eval_loader, DEVICE, as_tensor=True)
probe_ref_q      = reference_quantiles(probe_ref_losses, CFG.n_quantiles, DEVICE)
probe_member     = acc_and_loss(probe_model, probe_eval_loader, DEVICE)
probe_hold       = acc_and_loss(probe_model, probe_holdout_loader, DEVICE)
theta0_on_probe  = acc_and_loss(model, probe_eval_loader, DEVICE)

print(f"\n[probe] fit in {timings['probe_fit']:.1f}s")
print(f"[probe] on D_r^probe:  acc={probe_member['acc']:.2f}  loss={probe_member['loss']:.4f}")
print(f"[probe] unseen retain: acc={probe_hold['acc']:.2f}  loss={probe_hold['loss']:.4f}")
print(f"[probe] theta_0 on the same subset: acc={theta0_on_probe['acc']:.2f}  "
      f"loss={theta0_on_probe['loss']:.4f}")
_ = evaluate_all(probe_model, eval_loaders, DEVICE, "probe (sanity)")

if probe_member["loss"] > 3.0 * max(theta0_on_probe["loss"], 1e-6):
    print("\nWARNING: the probe is much weaker than theta_0 on D_r. With "
          "retain_mode='two_sided' this reference will actively pull retain accuracy "
          "down. Increase probe_steps/probe_size, or keep retain_mode='one_sided'.")

## 12. Sampling floors and $\epsilon_r$

`floor_r` is the irreducible $W_2^2$ for a `batch_retain`-sized draw from the probe reference;
$\epsilon_r$ = `eps_slack` × `floor_r`. Setting it below the floor makes sampling noise alone violate the
constraint and $\lambda$ diverge. `floor_f` is not a constraint — it is the level below which the forget term
has effectively converged, which is what you want to see in the top-right diagnostics panel.

In [ ]:
one_sided = (CFG.retain_mode == "one_sided")
floor_r = estimate_w2_floor(probe_ref_losses, CFG.batch_retain, CFG.n_quantiles,
                            one_sided=one_sided, seed=CFG.seed)
cal_theta0_losses = per_sample_losses(model, cal_loader, DEVICE, as_tensor=True)
floor_f = estimate_w2_floor(cal_theta0_losses, CFG.batch_forget, CFG.n_quantiles,
                            seed=CFG.seed)

eps_retain = CFG.eps_slack * floor_r if CFG.eps_retain < 0 else CFG.eps_retain
print(f"[floor] retain(batch={CFG.batch_retain}) = {floor_r:.4f}   "
      f"forget(batch={CFG.batch_forget}) = {floor_f:.4f}")
print(f"[floor] eps_retain = {eps_retain:.4f}  "
      f"(W2_f below ~{floor_f:.4f} means the forget term has converged)")

## 13. Run

In [ ]:
select_loaders = None
if CFG.select_best:
    rng = np.random.default_rng(CFG.seed)
    sel_f = np.sort(rng.choice(f_idx, min(CFG.select_n, len(f_idx)), replace=False))
    sel_r = np.sort(rng.choice(probe_idx, min(CFG.select_n, len(probe_idx)), replace=False))
    select_loaders = {"Df": make_loader(train_base, sel_f, 256, False),
                      "cal": cal_loader,
                      "Rprobe": make_loader(train_base, sel_r, 256, False)}

t0 = time.perf_counter()
model_w2, diag, plog, best = run_w2_unlearn(
    model, DEVICE, forget_loader, retain_loader, cal_loader, probe_ref_q,
    eps_retain, eval_loaders=eval_loaders, select_loaders=select_loaders)
timings["optimisation"] = time.perf_counter() - t0
print(f"optimisation: {timings['optimisation']:.1f}s")

## 14. Evaluate

In [ ]:
t0 = time.perf_counter()
results["w2probe"] = evaluate_all(model_w2, eval_loaders, DEVICE, "W2-probe matched")
mia["w2probe"]     = run_mia(model_w2, "W2-probe matched")
timings["evaluation"] = time.perf_counter() - t0

lf0   = per_sample_losses(model,    ev_forget,          DEVICE)
lf1   = per_sample_losses(model_w2, ev_forget,          DEVICE)
la1   = per_sample_losses(model_w2, ev_adjacent,        DEVICE)
lu1   = per_sample_losses(model_w2, ev_fish_holdout,    DEVICE)
lcal1 = per_sample_losses(model_w2, cal_loader,         DEVICE)
lrp1  = per_sample_losses(model_w2, probe_eval_loader,  DEVICE)
lref  = probe_ref_losses.numpy()

print(f"\nmedian loss  Df: {np.median(lf0):.3f} (before) -> {np.median(lf1):.3f} (after)")
print(f"             Dadj={np.median(la1):.3f}   unseen fish={np.median(lu1):.3f}")
print("distributional gaps (lower = closer to the counterfactual)")
print(f"  W2^2(Df, D_cal   [in objective]) = {w2_numpy(lf1, lcal1):.4f}")
print(f"  W2^2(Df, fish-te [held out]    ) = {w2_numpy(lf1, lu1):.4f}"
      f"   KS = {ks_numpy(lf1, lu1):.3f}")
print(f"  W2^2(Dr^probe under theta, probe ref) = {w2_numpy(lrp1, lref):.4f}")
print(f"  AUC(Df, unseen) = {mia['w2probe']['auc_unseen']:.3f} (0.5 = ideal)")

In [ ]:
if CFG.make_plots:
    save_diagnostics_plot(diag, plog, eps_retain, float(lref.mean()),
                          os.path.join(CFG.plot_dir, "w2probe_diagnostics.png"))
    save_loss_hist_plot(lf1, la1, lu1, lrp1, lref,
                        os.path.join(CFG.plot_dir, "w2probe_loss_hist.png"))

## 15. Results and runtime

In [ ]:
print_comparison(results, mia, timings)

In [ ]:
payload = {
    "config": {k: v for k, v in vars(CFG).items() if not k.startswith("__")},
    "eps_retain": eps_retain, "floor_r": floor_r, "floor_f": floor_f,
    "probe": {"size": int(len(probe_idx)), "n_adj_in_probe": n_adj_in_probe,
              "steps": CFG.probe_steps, "reinit": CFG.probe_reinit,
              "acc_member": probe_member["acc"], "loss_member": probe_member["loss"],
              "acc_holdout": probe_hold["acc"], "theta0_loss": theta0_on_probe["loss"]},
    "best_select": {k: v for k, v in best.items() if k != "state"},
    "timings": timings,
    "accuracy": {k: {kk: vv["acc"] for kk, vv in v.items()} for k, v in results.items()},
    "mia": mia,
}
with open(CFG.results_json, "w") as fh:
    json.dump(payload, fh, indent=2, default=str)
print(f"saved -> {CFG.results_json}")

os.makedirs(os.path.dirname(CFG.output) or ".", exist_ok=True)
torch.save({"model_state_dict": model_w2.state_dict(),
            "best": {k: v for k, v in best.items() if k != "state"}}, CFG.output)
with open(CFG.output.replace(".pt", "_diag.json"), "w") as fh:
    json.dump({"diag": diag, "probe": plog}, fh)
print(f"saved -> {CFG.output}")

## 16. Ablations

Change the flag in `CFG` and re-run from section 11 (or 13 when the probe is unchanged — it is cached by subset hash).

| Question | Setting |
|---|---|
| Does the whole refinement stack matter? | `CFG.literal_objective = True` |
| Is the one-sided retain term load-bearing? | `CFG.retain_mode = "two_sided"` |
| Does the floor calibration matter? | `CFG.eps_slack` over `[0.5, 1.5, 5, 20]` |
| Does head re-initialisation matter? | `CFG.probe_reinit = "none"` |
| How cheap can FitCheap get? | `probe_reinit="head"`, `probe_steps=100`, `probe_size=2000` |
| Does calibration pooling matter? | `CFG.cal_pool = CFG.batch_forget`, `CFG.cal_refresh = 1` |
| Does checkpoint selection matter? | `CFG.select_best = False` |
| Variance | vary `CFG.seed` / `CFG.split_seed` ($\lvert D_f\rvert=250$) |